# DistilBERT · Phát hiện tin giả về COVID-19
**Lương Hoàng Bảo Quân · 24DH111520 · Nhóm 5, HUFLIT**

DistilBERT đọc bài đăng tiếng Anh và chọn nhãn `fake` hoặc `real`. Ba lần huấn luyện dùng cùng dữ liệu và cấu hình. **Seed 123** có F1 weighted cao nhất trên Validation. **Seed 42** là checkpoint dùng trong phiên Kaggle đã lưu, có token hóa, logits, đường loss, phân tích lỗi và lượt huấn luyện lại.

Mở trên GitHub để đọc ngay các output. Chọn **Run all** với `MODE = "xem"` trong Colab, Kaggle hoặc Jupyter để dựng lại các bảng từ tệp CSV và JSON. Các chế độ dùng mô hình nằm ở cuối notebook.

[Phiên Kaggle gốc và output từng ô](minh_chung/Phien_Kaggle_seed42.ipynb) · [Báo cáo ba seed](../../reports/distilbert) · [Trang bài nộp](../../README.md)

## 1. Chọn chế độ
Giữ `xem` để trình bày. Chọn `du_doan` để nạp checkpoint đã huấn luyện và thử bộ 10 câu chung, `train` để huấn luyện đầy đủ một seed, hoặc `smoke` để thử quy trình trên ít mẫu. Có thể mở notebook bên trong ZIP đã giải nén để xem kết quả ngoại tuyến. Khi mở riêng notebook trên Colab, ô dưới tải repo của nhóm.

In [1]:
from pathlib import Path
from collections import Counter
from statistics import mean, stdev
import csv
import hashlib
import json
import os
import subprocess
import sys
import tempfile

MODE = "xem"                  # xem | du_doan | train | smoke
SEED = 42                     # 42 | 123 | 2024
PROJECT_DIR = None            # Có thể chỉ định nơi đã giải nén repo
MODEL_DIR = None              # Thư mục model/ của đúng seed, cạnh run_config.json
INSTALL_DEPS = False          # Bật khi phiên mới cần cài thư viện cho mô hình

def is_project(path):
    return (path / "src/finetune.py").is_file() and (path / "reports/distilbert/seed42/metrics.json").is_file()

current = Path.cwd()
workdir = next((p for p in [Path("/kaggle/working"), Path("/content")] if p.is_dir()), current)
if PROJECT_DIR:
    project = Path(PROJECT_DIR).expanduser().resolve()
else:
    candidates = [current, *current.parents, workdir / "covid19-fake-news-nhom5"]
    project = next((p for p in candidates if is_project(p)), None)
    if project is None and Path("/kaggle/input").is_dir():
        project = next((p.parent.parent for p in Path("/kaggle/input").rglob("configs/distilbert.json")
                        if is_project(p.parent.parent)), None)
    if project is None:
        project = workdir / "covid19-fake-news-nhom5"
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                        "https://github.com/Doris1808/covid19-fake-news-nhom5.git", str(project)], check=True)
if not is_project(project):
    raise FileNotFoundError("Đặt PROJECT_DIR tới thư mục repo chứa src, data và reports.")

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

config = read_json(project / "configs/distilbert.json")
if MODE not in {"xem", "du_doan", "train", "smoke"} or SEED not in config["seeds"]:
    raise ValueError("Kiểm tra MODE và SEED ở đầu ô.")
reports_dir = project / "reports/distilbert"
data_dir = project / "data/raw"
print(f"Chế độ {MODE}  |  Seed chọn {SEED}  |  Mô hình {config['model_name']}")

Chế độ xem  |  Seed chọn 42  |  Mô hình distilbert/distilbert-base-uncased


## 2. Dữ liệu và cách mô hình học
Tokenizer tách bài đăng thành token, thêm token đặc biệt và cắt ở 128 token. Khi train, bộ nạp đệm từng batch. Mô hình trả hai logits, tính cross-entropy theo nhãn `fake=0` và `real=1`. Bốn batch 8 mẫu cho một bước cập nhật tương đương 32 mẫu trên một GPU. Ô này đếm nhãn trong ba tập và đối chiếu SHA-256 với hồ sơ từng seed.

In [2]:
def csv_rows(path):
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))

hashes = {}
print(f"{'Tập':<12}{'Fake':>8}{'Real':>8}{'Tổng':>8}")
for split in ("train", "val", "test"):
    path = data_dir / f"{split}.csv"
    counts = Counter(row["label"].strip().lower() for row in csv_rows(path))
    if set(counts) != {"fake", "real"}:
        raise ValueError(f"Nhãn lạ trong {split}.")
    hashes[split] = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{split:<12}{counts['fake']:>8}{counts['real']:>8}{sum(counts.values()):>8}")
for seed in config["seeds"]:
    folder = reports_dir / f"seed{seed}"
    protocol = read_json(folder / "run_config.json")
    test = read_json(folder / "test_metrics.json")
    expected = {**protocol["dataset_sha256"], "test": test["dataset_sha256"]}
    if expected != hashes or protocol["seed"] != seed or protocol["smoke"]:
        raise ValueError(f"Dữ liệu hoặc loại lượt chạy khác báo cáo seed {seed}.")
    if any(protocol.get(k) != v for k, v in config.items()):
        raise ValueError(f"Cấu hình seed {seed} khác cấu hình chung.")
print("Đã đối chiếu SHA-256 của ba tập cho cả ba seed.")
print(f"{config['epochs']} epoch · learning rate {config['learning_rate']} · max_length {config['max_length']}")

Tập             Fake    Real    Tổng
train           3060    3360    6420
val             1020    1120    2140
test            1020    1120    2140
Đã đối chiếu SHA-256 của ba tập cho cả ba seed.
4 epoch · learning rate 2e-05 · max_length 128


## 3. Kết quả của ba seed
Mỗi lượt dùng checkpoint có **F1 weighted cao nhất trên Validation** để đánh giá Test. Bảng đọc từ `metrics.json` và `test_metrics.json` của từng lượt. F1w là F1 weighted, F1m là F1 macro. Các điểm có thang 0–1. Độ lệch chuẩn mẫu dùng mẫu số `n − 1`.

In [3]:
records = []
for seed in config["seeds"]:
    folder = reports_dir / f"seed{seed}"
    training = read_json(folder / "metrics.json")
    test = read_json(folder / "test_metrics.json")
    records.append((seed, training["val"]["f1_weighted"], test["accuracy"], test["f1_weighted"],
                    test["f1_macro"], training["train_seconds_this_session"]))
print(f"{'Seed':>5}  {'Val F1w':>9}  {'Test Acc':>9}  {'Test F1w':>9}  {'Test F1m':>9}  {'Train (s)':>9}")
for seed, val_w, acc, test_w, test_m, seconds in records:
    print(f"{seed:5d}  {val_w:9.5f}  {acc:9.5f}  {test_w:9.5f}  {test_m:9.5f}  {seconds:9.1f}")
chosen_seed = max(records, key=lambda row: row[1])[0]
weighted = [row[3] for row in records]
print(f"Chọn theo Validation: seed {chosen_seed}")
print(f"Test F1 weighted qua ba seed: {mean(weighted):.5f} ± {stdev(weighted):.5f}")

 Seed    Val F1w   Test Acc   Test F1w   Test F1m  Train (s)
   42    0.97195    0.96963    0.96962    0.96954      246.8
  123    0.97522    0.97290    0.97289    0.97283      251.4
 2024    0.97150    0.97243    0.97244    0.97238      251.2
Chọn theo Validation: seed 123
Test F1 weighted qua ba seed: 0.97165 ± 0.00177


## 4. Lỗi trên Test và hai câu ngoài bộ dữ liệu
Đọc theo hàng là nhãn thật, theo cột là dự đoán. Với seed 42, mô hình bỏ sót 39 trong 1.020 bài mang nhãn fake và gán nhầm 26 trong 1.120 bài mang nhãn real. Đổi `SEED` ở ô đầu để xem ma trận của lượt khác.

Nhóm còn lưu hai câu có nhãn tham chiếu lấy từ WHO cho **seed 123**. Hai câu này minh họa lỗi khi dùng mô hình ngoài tập Test. Ô cuối cho phép thử thêm bộ 10 câu chung của nhóm.

In [4]:
test_record = read_json(reports_dir / f"seed{SEED}" / "test_metrics.json")
(a, b), (c, d) = test_record["confusion_matrix_fake_real"]
print(f"Test · seed {SEED}         Dự đoán fake   Dự đoán real")
print(f"Nhãn thật fake            {a:>11}    {b:>11}")
print(f"Nhãn thật real            {c:>11}    {d:>11}")
print(f"Sai {b+c}/{a+b+c+d} bài · Recall lớp fake = {a/(a+b):.4f}")
print()
for row in csv_rows(reports_dir / "external_examples_seed123.csv"):
    print(f"{row['id']}. {row['text']}")
    print(f"   Nhãn tham chiếu {row['reference_label']} | dự đoán {row['predicted_label']} | seed {row['model_seed']}")

Test · seed 42         Dự đoán fake   Dự đoán real
Nhãn thật fake                    981             39
Nhãn thật real                     26           1094
Sai 65/2140 bài · Recall lớp fake = 0.9618

1. People of all ages can get infected by the new coronavirus.
   Nhãn tham chiếu real | dự đoán fake | seed 123
2. No, drinking alcohol does not protect you from COVID-19 infection.
   Nhãn tham chiếu real | dự đoán fake | seed 123


## 5. Chạy mô hình
Chọn GPU khi huấn luyện. Trong phiên mới, bật `INSTALL_DEPS` nếu cần cài thư viện. Lượt `train` tạo thư mục mới, huấn luyện một seed và đánh giá Test sau khi chọn checkpoint trên Validation. Lượt `smoke` kiểm tra trên ít mẫu trong một epoch. Để dùng `du_doan`, gắn thư mục chứa `model/` và `run_config.json`. Nếu có nhiều bản cùng seed, chỉ định `MODEL_DIR` cụ thể.

[Đọc code huấn luyện](../../src/finetune.py) · [Đọc code đánh giá](../../src/evaluate.py) · [Xem output thực tế của lượt Kaggle seed 42](minh_chung/Phien_Kaggle_seed42.ipynb)

In [5]:
def checkpoint_for_seed():
    if MODEL_DIR:
        options = [Path(MODEL_DIR).expanduser().resolve()]
    else:
        options = []
        for directory in (project / "outputs", workdir / "distilbert_runs", Path("/kaggle/input")):
            if directory.is_dir():
                options.extend(p.parent / "model" for p in directory.rglob("run_config.json"))
    matches = []
    for folder in dict.fromkeys(options):
        record = folder.parent / "run_config.json"
        if record.is_file() and (folder / "config.json").is_file():
            info = read_json(record)
            has_weights = any(folder.glob("*.safetensors")) or (folder / "pytorch_model.bin").is_file()
            same_config = all(info.get(k) == v for k, v in config.items())
            if (has_weights and same_config and info.get("seed") == SEED and info.get("smoke") is False
                and info.get("dataset_sha256") == {k: hashes[k] for k in ("train", "val")}):
                matches.append(folder)
    if len(matches) != 1:
        raise ValueError(f"Tìm thấy {len(matches)} checkpoint hợp lệ cho seed {SEED}. Đặt MODEL_DIR tới đúng model/.")
    return matches[0]

if MODE != "xem":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    if INSTALL_DEPS:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(project / "requirements.txt")], check=True)
    if str(project) not in sys.path:
        sys.path.insert(0, str(project))
    if MODE in {"smoke", "train"}:
        run_root = workdir / "distilbert_runs"
        run_root.mkdir(parents=True, exist_ok=True)
        run_dir = Path(tempfile.mkdtemp(prefix=f"{MODE}_seed{SEED}_", dir=run_root))
        command = [sys.executable, "-m", "src.finetune", "--seed", str(SEED), "--out-dir", str(run_dir)]
        if MODE == "smoke":
            command.append("--smoke")
        subprocess.run(command, cwd=project, check=True)
        if MODE == "train":
            subprocess.run([sys.executable, "-m", "src.evaluate", "--run-dir", str(run_dir),
                            "--confirm-config-frozen"], cwd=project, check=True)
            model_dir = run_dir / "model"
        print("Thư mục của lượt chạy:", run_dir)
    else:
        model_dir = checkpoint_for_seed()
        print("Checkpoint:", model_dir)
else:
    print("Đang xem báo cáo đã lưu. Chọn chế độ ở ô đầu để dùng mô hình.")

Đang xem báo cáo đã lưu. Chọn chế độ ở ô đầu để dùng mô hình.


### Dự đoán 10 câu chung
Ở chế độ `du_doan` hoặc sau khi `train` đầy đủ, ô này tính nhãn dự đoán cho từng câu rồi xuất CSV kèm seed, nhãn tham chiếu và điểm của hai lớp. Mỗi lần chạy lưu kết quả ở thư mục mới.

In [6]:
if MODE in {"du_doan", "train"}:
    from src.predict import load_model, predict
    tokenizer, model = load_model(model_dir)
    labelled = csv_rows(project / "data/external_10_sentences.csv")
    assert len(labelled) == 10 and len({r["id"] for r in labelled}) == 10
    results = []
    for row in labelled:
        probabilities = predict(row["sentence"], tokenizer, model, config["max_length"])
        predicted = max(probabilities, key=probabilities.get)
        results.append({**row, "seed": SEED, "predicted": predicted, "score_fake": probabilities["fake"],
                        "score_real": probabilities["real"], "correct": predicted == row["label"].strip().lower()})
    result_dir = Path(tempfile.mkdtemp(prefix=f"external10_seed{SEED}_", dir=workdir))
    destination = result_dir / "external_10_results.csv"
    with destination.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(results[0]))
        writer.writeheader()
        writer.writerows(results)
    print(f"Đúng {sum(r['correct'] for r in results)}/{len(results)} câu. Kết quả: {destination}")
else:
    print("Bộ 10 câu chung nằm trong data/external_10_sentences.csv.")

Bộ 10 câu chung nằm trong data/external_10_sentences.csv.
